# Preparation exercise
**Learning from data [TIF285], Chalmers, Fall 2026**  
*Last modified: 2026-08-30*

This is a standalone exercise to be completed *before* the first problem set.
It is not graded. Its purpose is to confirm that your python environment is
working and that you can run, edit and submit a Jupyter notebook, so that any
problems surface now rather than the day before a deadline.

Work through the two parts below in order.

<a id="installations"></a>
## Part 1: Installations

Perform the installations and preparations that are described in the Getting Started instructions. At the end you should have:

1. downloaded the current version of the course material from the github repository or from the course web page;
2. a running python installation that includes the modules listed in the environment.yml file (e.g. numpy, matplotlib, pandas, emcee, scikit-learn, ...);
3. been able to open and run the Jupyter Notebooks with the first week exercises.
Ask the computer lab supervisors for assistance if needed.

In [ ]:
# Modules needed for tests and file system operations
import sys

import os

# Where to save the figures and data files
DATA_ID = "DataFiles/"
if not os.path.exists(DATA_ID):
    os.makedirs(DATA_ID)

# Make sure that you are running python with version >= 3.x
#
# Import the following python modules with
# the specified abreviations:
# ---
# numpy as np
# scipy as scipy
# scipy.stats as stats
# pandas as pd
# matplotlib.pyplot as plt
# sklearn as skl
# emcee as emcee


##################
# YOUR CODE HERE #
##################


In [ ]:
# TESTS TO CHECK YOUR CODE
# ---
assert sys.version_info.major>=3, \
    'You are running Python version'+\
    f'{sys.version_info.major}.{sys.version_info.minor}'

modules = [('numpy','np'), ('scipy', 'scipy'), \
           ('pandas', 'pd'), ('matplotlib.pyplot', 'plt'), \
           ('sklearn', 'skl'), ('emcee', 'emcee')]
for (_module, _module_abbrev) in modules:
    assert _module in sys.modules and _module_abbrev in dir(),\
        f'Module {_module} not loaded properly.'

<a id="randomsamples"></a>
## Part 2: Correlated random variables

[Ortho-positronium](https://en.wikipedia.org/wiki/Positronium), the
spin-triplet bound state of an electron and a positron, cannot annihilate into
two photons. Charge-conjugation symmetry forbids it. Instead, the state decays into
three photons,

$$
\mathrm{o\text{-}Ps} \rightarrow \gamma\gamma\gamma .
$$

Consider such annihilation events at rest. For each event, the
$x$-component of the momentum of each of the three photons is measured and recorded,

$$
(P_1, P_2, P_3) \equiv (p_{1x}, p_{2x}, p_{3x}).
$$

Since the positronium is at rest, momentum conservation requires that in
**every** event

$$
P_1 + P_2 + P_3 = 0 .
$$

We model the three components as a multivariate Gaussian with zero mean. (The
true photon spectrum is not Gaussian, but that does not matter here: what this
exercise is about is the *constraint*, and the Gaussian is a convenient stand-in
for the shape of the distribution.) We can measure the momenta in units
of their common standard deviation $\sigma_p$, so that each univariate marginal
distribution is a standard normal ($\mu = 0$, $\sigma^2 = 1$). Because the three
photons are identical, the covariance matrix must be unchanged under any
relabelling of them, which leaves only a single free parameter $\rho$:

$$
(P_1, P_2, P_3) \sim \mathcal{N}\left( \boldsymbol{\mu}, \boldsymbol{\Sigma} \right),
\qquad
\boldsymbol{\mu} = (0,0,0), \qquad
\boldsymbol{\Sigma} = \left(
\begin{array}{ccc}
1 & \rho & \rho \\
\rho & 1 & \rho \\
\rho & \rho & 1
\end{array}
\right).
$$

Take a minute to convince yourself of the following three statements, which
together explain why this problem is a natural example of *anticorrelated*
random variables.

1. **The constraint fixes $\rho$.** Momentum conservation
   demands that $P_1+P_2+P_3=0$. This result must hold with zero variance. Using
   $\mathrm{Var}(P_1+P_2+P_3) = \sum_i \mathrm{Var}(P_i)
   + 2\sum_{i<j}\mathrm{Cov}(P_i,P_j) = 3(1 + 2\rho)$, so

   $$\rho = -\tfrac{1}{2}.$$

   Momentum conservation *is* the anticorrelation: if one photon happens to fly
   off in the $+x$ direction, the other two must, between them, balance it.

2. **This is the most negative $\rho$ can be.** The eigenvalues of
   $\boldsymbol{\Sigma}$ are $1+2\rho$ (once, with eigenvector
   $\propto(1,1,1)$) and $1-\rho$ (twice). A covariance matrix must be positive
   semi-definite, so $\rho \ge -1/2$ is required. At $\rho = -1/2$ the matrix is
   singular, and the singular direction is exactly $(1,1,1)$ -- the conserved
   sum, along which there is no spread at all.

3. **The distribution is degenerate, and that is the point.** Because
   $\boldsymbol{\Sigma}$ is singular, the samples do not fill three-dimensional
   space: every event lies exactly on the plane $P_1+P_2+P_3=0$, which is a
   two-dimensional subspace. Equivalently, $\boldsymbol{\Sigma}$ has rank 2
   rather than rank 3.

   Be careful about what this does *not* say. It does not mean that two of the
   components are independent and the third is redundant: all three remain
   pairwise anticorrelated, with $\rho = -1/2$ between every pair. It says only
   that once any two components are known the third is fixed, so the support of
   the distribution is a plane rather than a volume.

   None of this is a numerical obstacle. `numpy` samples a multivariate normal
   through a singular value decomposition of $\boldsymbol{\Sigma}$, which
   handles the singular direction correctly, and you will find that the three
   components of each generated event sum to zero to machine precision.

**Task (see also Yata):** Write a function
`sample_momenta(mean, cov, size=6, seed=2026)`
that returns an array of shape `(size, 3)` containing samples of
$(P_1, P_2, P_3)$. Use the `multivariate_normal` method of a `numpy` random
`Generator`. 

*Important*: create the generator inside the function with
`numpy.random.default_rng` with the seed as input (default 2026), so that
repeated calls with the same arguments return the same samples.

As discussed in the Statistics chapter of the lecture notes, many standard
distributions are built into `numpy` itself, so when all you need is to draw
samples you can often use `numpy` directly rather than going through
`scipy.stats`.
We do so deliberately here: the `scipy.stats` route uses `numpy` underneath
but adds a layer whose output for this distribution has changed between
`scipy` versions, which would make your results depend on which version you
happen to have installed.

In [ ]:
def sample_momenta(mean, cov, size=6, seed=2026):
    '''
    Draw samples of the three photon momentum components.

    Args:
        mean: array_like, shape (3,)
            Mean of the multivariate normal.
        cov: array_like, shape (3,3)
            Covariance matrix of the multivariate normal.
        size: int, default 6
            Number of events (rows) to generate.
        seed: int, default 2026
            Seed for the RNG

    Returns:
        ndarray, shape (size, 3): the sampled momentum components.
    '''
##################
# YOUR CODE HERE #
##################


Call your function with the $\boldsymbol{\mu}$ and $\boldsymbol{\Sigma}$ defined above to generate 6 events. Store the result in a variable called `momenta` and print it.

In [ ]:
# The mean and covariance derived above
mean = [0, 0, 0]
rho = -0.5
cov = rho*np.ones((3, 3))
np.fill_diagonal(cov, 1)

##################
# YOUR CODE HERE #
##################


In [ ]:
# TESTS TO CHECK YOUR CODE
# ---
assert np.shape(momenta) == (6, 3), \
    'The result should have six rows (events) and three columns'
assert np.allclose(momenta, sample_momenta(mean, cov, size=6)), \
    'Repeated calls with the same arguments should give the same samples. ' \
    'Create the generator inside the function with a fixed seed.'

**Task:** Create a `Pandas.DataFrame` from the sample array `momenta`.

The columns should be labelled `['P1', 'P2', 'P3']` and the row indices `['Event 1', 'Event 2', ...]`. Add a further column `'sum'` holding the sum of the three components in each event. Print the DataFrame (you can try with `print` and with the fancier `display`).

Compare the entries in the `'sum'` column with the individual momentum components. What do you find, and what does it tell you about where in the three-dimensional space $(P_1, P_2, P_3)$ the generated events lie?

In [ ]:
##################
# YOUR CODE HERE #
##################


<a id="concepts"></a>
## Part 3: Concept questions

The two questions below are answered on Yata. No code is needed for either.

**Question 1 (see also Yata).** You generated the events as rows of three numbers. Think of each
event instead as a single point in the three-dimensional space whose axes are
$P_1$, $P_2$ and $P_3$. If you generated a very large number of events and
plotted them all, they would fill

* a three-dimensional cloud, spread through the whole space
* a two-dimensional plane
* a one-dimensional line
* a single point

*Hint*: you already have the evidence for this in the `'sum'` column.

**Question 2 (see also Yata).** Positronium can also annihilate into **four** photons, although
this happens far more rarely. Repeat the argument of statement 1 above for a
decay at rest into four identical particles: exchange symmetry again leaves a
single correlation $\rho$ shared by every pair of momentum components. What is
its value?

* $\rho = -3/4$
* $\rho = -1/2$
* $\rho = +1/2$
* $\rho = -1/3$
* $\rho = -1/4$